# EDA 11: District HQ points (Wikidata) and climate (NASA POWER) — load, tidy, map to LGD

**Sources:** `wikidata_hq` (CC0) for each district headquarters' coordinates, looked up
**by LGD district code** (Wikidata property P12746), and `nasa_power` (CC BY 4.0) daily
point data read at those 75 points. No boundaries needed (design doc §6, Climate).

**Question:** Can we get one climate and solar record per district, keyed by LGD code, for
enough years to show variability (rainfall, heat) and irradiance?

| | |
|---|---|
| **Inputs** | Wikidata SPARQL endpoint; NASA POWER `/api/temporal/daily/point` (no key). Or the raw files already in `data/raw/{wikidata_hq,nasa_power}/ingest_date=*/` |
| **Outputs** | `data/interim/climate/hq_points.parquet` (75 rows); `data/interim/climate/power_daily.parquet` (one row per district per day, 2001–2025) |
| **Grain** | point (HQ) per district; daily |
| **Run** | *Restart and Run All*. Fetches only when no snapshot exists; `UP_SITING_CLIMATE_REFETCH=1` fetches a new one (≈2 minutes) |

In [1]:
import io
import json
import os
import time
import urllib.parse
import urllib.request
from datetime import datetime
from zoneinfo import ZoneInfo

import duckdb
import pandas as pd

from up_siting.common.paths import INTERIM_DIR, RAW_DIR, SEEDS_DIR
from up_siting.extract.manifest import register_file

USER_AGENT = "up-siting-eda/0.1 (district siting research notebook)"  # Wikidata asks for one
SPARQL = "https://query.wikidata.org/sparql"
POWER = "https://power.larc.nasa.gov/api/temporal/daily/point"
PARAMS = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "ALLSKY_SFC_SW_DWN", "RH2M"]
START, END = "20010101", "20251231"  # CERES irradiance (ALLSKY) starts in 2001
REFETCH = os.environ.get("UP_SITING_CLIMATE_REFETCH") == "1"
TODAY = datetime.now(ZoneInfo("Asia/Kolkata")).date().isoformat()
OUT_DIR = INTERIM_DIR / "climate"

spine = pd.read_csv(SEEDS_DIR / "lgd_district_spine.csv")
print(len(spine), "districts")


def newest(source: str, pattern: str):
    # Newest ingest_date folder of `source` containing a file matching `pattern`, else None.
    for folder in sorted((RAW_DIR / source).glob("ingest_date=*"), reverse=True):
        if any(folder.glob(pattern)):
            return folder
    return None


def get(url: str, timeout: int = 120) -> bytes:
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read()

75 districts


## 1. District HQs from Wikidata, by LGD code

The query passes our 75 LGD codes in a `VALUES` clause, so the join to the spine happens in
the query itself: no name matching. For each district it asks for its capital (P36) and
the capital's coordinates (P625), plus the district item's own coordinates as a fallback.

In [2]:
codes = " ".join(f'"{c}"' for c in spine["lgd_district_code"])
QUERY = f"""
SELECT ?lgd ?district ?districtLabel ?hq ?hqLabel ?hq_coord ?district_coord WHERE {{
  VALUES ?lgd {{ {codes} }}
  ?district wdt:P12746 ?lgd .
  OPTIONAL {{ ?district wdt:P36 ?hq . OPTIONAL {{ ?hq wdt:P625 ?hq_coord }} }}
  OPTIONAL {{ ?district wdt:P625 ?district_coord }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}"""

hq_dir = newest("wikidata_hq", "wikidata_hq.json")
if hq_dir is None or REFETCH:
    hq_dir = RAW_DIR / "wikidata_hq" / f"ingest_date={TODAY}"
    hq_dir.mkdir(parents=True, exist_ok=True)
    url = SPARQL + "?" + urllib.parse.urlencode({"query": QUERY, "format": "json"})
    (hq_dir / "wikidata_hq.json").write_bytes(get(url))
    (hq_dir / "wikidata_hq.rq").write_text(QUERY, encoding="utf-8")
    for name in ["wikidata_hq.json", "wikidata_hq.rq"]:
        register_file(
            "wikidata_hq",
            hq_dir / name,
            SPARQL,
            fetched_by="extractor",
            notes="SPARQL by LGD District Code (P12746); the .rq file is the query",
        )

bindings = json.loads((hq_dir / "wikidata_hq.json").read_text(encoding="utf-8"))
rows = [{k: v["value"] for k, v in b.items()} for b in bindings["results"]["bindings"]]
wd = pd.DataFrame(rows)
print(hq_dir.name, len(wd), "rows for", wd["lgd"].nunique(), "LGD codes")

ingest_date=2026-11-24 91 rows for 75 LGD codes


In [3]:
assert set(wd["lgd"].astype(int)) == set(spine["lgd_district_code"]), "a district is missing"

## 2. One point per district

Rows repeat when an item has two coordinate statements (e.g. Rampur, Pratapgarh). Rule:
take the **median** of the HQ's coordinates. Fall back to the district item's own
coordinates, and flag it, when the HQ has no coordinates **or** its point is implausible:
outside UP's bounding box or more than 60 km from the district's own point (a wrongly
linked HQ item; see the output). Then check every chosen point.

In [4]:
def parse_point(wkt):
    # 'Point(lon lat)' -> (lat, lon)
    if not isinstance(wkt, str):
        return (None, None)
    lon, lat = wkt.removeprefix("Point(").removesuffix(")").split()
    return (float(lat), float(lon))


wd[["hq_lat", "hq_lon"]] = wd["hq_coord"].apply(lambda w: pd.Series(parse_point(w)))
wd[["d_lat", "d_lon"]] = wd["district_coord"].apply(lambda w: pd.Series(parse_point(w)))
pts = (
    wd.groupby("lgd")
    .agg(
        wikidata_district=("district", "first"),
        wikidata_label=("districtLabel", "first"),
        hq_label=("hqLabel", "first"),
        n_hq=("hq", "nunique"),
        hq_lat=("hq_lat", "median"),
        hq_lon=("hq_lon", "median"),
        d_lat=("d_lat", "median"),
        d_lon=("d_lon", "median"),
    )
    .reset_index()
)
# great-circle distance HQ point <-> district item point (km)
pts["hq_km_to_district_point"] = (
    duckdb.sql("""
    select 6371 * 2 * asin(sqrt(
        pow(sin(radians(d_lat - hq_lat) / 2), 2) +
        cos(radians(hq_lat)) * cos(radians(d_lat)) * pow(sin(radians(d_lon - hq_lon) / 2), 2)
    )) as km from pts
""")
    .df()["km"]
    .round(1)
)
in_up = pts["hq_lat"].between(23.8, 30.5) & pts["hq_lon"].between(77.0, 84.7)
hq_ok = pts["hq_lat"].notna() & in_up & ~(pts["hq_km_to_district_point"] > 60)
pts["point_kind"] = "hq"
pts.loc[pts["hq_lat"].isna(), "point_kind"] = "district_item (hq has no coordinates)"
pts.loc[pts["hq_lat"].notna() & ~hq_ok, "point_kind"] = "district_item (hq point rejected)"
pts["lat"] = pts["hq_lat"].where(hq_ok, pts["d_lat"]).round(4)
pts["lon"] = pts["hq_lon"].where(hq_ok, pts["d_lon"]).round(4)
pts["lgd_district_code"] = pts["lgd"].astype(int)
pts = pts.merge(spine[["lgd_district_code", "district_name"]], on="lgd_district_code")
pts[(pts["point_kind"] != "hq") | (pts["n_hq"] > 1) | (pts["hq_km_to_district_point"] > 30)][
    [
        "district_name",
        "hq_label",
        "n_hq",
        "hq_lat",
        "hq_lon",
        "hq_km_to_district_point",
        "point_kind",
        "lat",
        "lon",
    ]
]

,district_name,hq_label,n_hq,hq_lat,hq_lon,hq_km_to_district_point,point_kind,lat,lon
14,Bijnor,Bijnor,1,29.370000,78.130000,37.8,hq,29.3700,78.1300
55,Pilibhit,Pilibhit,1,28.633333,79.766667,33.8,hq,28.6333,79.7667
64,Siddharthnagar,Navgarh,1,NaN,NaN,NaN,district_item (hq has no coordinates),27.2792,83.0546
71,Amethi,Gauriganj,1,26.433333,87.733333,592.2,district_item (hq point rejected),26.1689,81.7994


In [5]:
assert len(pts) == 75 and pts["lgd_district_code"].is_unique
assert pts["lat"].between(23.8, 30.5).all() and pts["lon"].between(77.0, 84.7).all()
assert (pts["point_kind"] == "hq").sum() >= 70
assert pts[["lat", "lon"]].notna().all().all()

## 3. NASA POWER daily data at each point

One request per district (6 parameters, 2001–2025, community RE so irradiance is in
kWh/m²/day). Each response is saved unchanged as CSV and registered. Requests are made
one at a time with a short pause, to be polite to a free service.

In [6]:
def power_url(lat: float, lon: float) -> str:
    q = {
        "parameters": ",".join(PARAMS),
        "community": "RE",
        "latitude": lat,
        "longitude": lon,
        "start": START,
        "end": END,
        "format": "CSV",
    }
    return POWER + "?" + urllib.parse.urlencode(q)


power_dir = newest("nasa_power", "power_daily_*.csv")
if power_dir is None or REFETCH:
    power_dir = RAW_DIR / "nasa_power" / f"ingest_date={TODAY}"
    power_dir.mkdir(parents=True, exist_ok=True)
    for p in pts.itertuples():
        path = power_dir / f"power_daily_{p.lgd_district_code}.csv"
        url = power_url(p.lat, p.lon)
        path.write_bytes(get(url, timeout=300))
        register_file(
            "nasa_power",
            path,
            url,
            fetched_by="extractor",
            notes=f"{p.district_name} HQ point ({p.point_kind})",
        )
        time.sleep(0.5)
files = sorted(power_dir.glob("power_daily_*.csv"))
print(power_dir.name, len(files), "files")

ingest_date=2026-11-24 75 files


In [7]:
assert len(files) == 75

## 4. Parse and tidy

Each CSV has a text header between `-BEGIN HEADER-` and `-END HEADER-` (units, the grid
cell's elevation), then the table. `-999` means missing.

In [8]:
def read_power_csv(path):
    text = path.read_text(encoding="utf-8")
    header, table = text.split("-END HEADER-\n", 1)
    df = pd.read_csv(io.StringIO(table))
    df["date"] = pd.to_datetime(dict(year=df["YEAR"], month=df["MO"], day=df["DY"]))
    df = df.drop(columns=["YEAR", "MO", "DY"]).replace(-999.0, pd.NA)
    df.insert(0, "lgd_district_code", int(path.stem.rsplit("_", 1)[1]))
    elevation = header.split("= ")[-1].split(" meters")[0] if "elevation" in header else None
    return df, header, elevation


parsed = [read_power_csv(f) for f in files]
daily = pd.concat([p[0] for p in parsed], ignore_index=True)
daily = daily.rename(
    columns={
        "T2M": "t2m_c",
        "T2M_MAX": "t2m_max_c",
        "T2M_MIN": "t2m_min_c",
        "PRECTOTCORR": "precip_mm",
        "ALLSKY_SFC_SW_DWN": "ghi_kwh_m2_day",
        "RH2M": "rh2m_pct",
    }
)
print(parsed[0][1])
daily.describe().T.round(2)

-BEGIN HEADER-
NASA/POWER Source Native Resolution Daily Data 
Dates (month/day/year): 01/01/2001 through 12/31/2025 in LST
Location: latitude  27.18   longitude 78.02 
elevation from MERRA-2: Average for 0.5 x 0.625 degree lat/lon region = 162.41 meters
The value for missing source data that cannot be computed or is outside of the sources availability range: -999 
parameter(s): 
T2M                   MERRA-2 Temperature at 2 Meters (C) 
T2M_MAX               MERRA-2 Temperature at 2 Meters Maximum (C) 
T2M_MIN               MERRA-2 Temperature at 2 Meters Minimum (C) 
PRECTOTCORR           MERRA-2 Precipitation Corrected (mm/day) 
ALLSKY_SFC_SW_DWN     CERES SYN1deg All Sky Surface Shortwave Downward Irradiance (kW-hr/m^2/day) 
RH2M                  MERRA-2 Relative Humidity at 2 Meters (%) 



,count,mean,min,25%,50%,75%,max,std
lgd_district_code,684825.0,185.706667,118.0,136.0,155.0,174.0,661.0,125.8086
t2m_c,684825.0,25.818673,6.54,19.64,27.31,30.96,42.38,7.235859
t2m_max_c,684825.0,32.666074,11.18,27.82,32.23,37.6,49.84,6.762875
t2m_min_c,684825.0,19.845611,-1.64,13.04,21.58,26.18,36.56,7.643472
precip_mm,684825.0,2.578125,0.0,0.0,0.0,1.23,279.27,7.375299
ghi_kwh_m2_day,684825.0,4.815768,0.2448,3.6787,4.8746,6.0504,8.6098,1.552832
rh2m_pct,684825.0,51.717727,3.81,32.34,50.27,73.16,96.01,23.333732
date,684825,2013-07-02 00:00:00,2001-01-01 00:00:00,2007-04-02 00:00:00,2013-07-02 00:00:00,2019-10-02 00:00:00,2025-12-31 00:00:00,NaN


In [9]:
n_days = (pd.Timestamp(END) - pd.Timestamp(START)).days + 1
assert len(daily) == 75 * n_days and not daily.duplicated(["lgd_district_code", "date"]).any()
missing = daily.drop(columns=["lgd_district_code", "date"]).isna().mean()
print(missing.round(4).to_dict())
assert (missing < 0.01).all()
assert daily["t2m_max_c"].between(-5, 52).all() and daily["precip_mm"].dropna().ge(0).all()
assert daily["ghi_kwh_m2_day"].dropna().between(0, 10).all()

{'t2m_c': 0.0, 't2m_max_c': 0.0, 't2m_min_c': 0.0, 'precip_mm': 0.0, 'ghi_kwh_m2_day': 0.0, 'rh2m_pct': 0.0}


## 5. How many distinct grid cells do 75 points really hit?

POWER's meteorology comes from MERRA-2 (0.5° × 0.625°) and irradiance from CERES (1°).
Neighbouring HQs can fall in the same cell and get identical series, so the climate
variable can't tell them apart. Count the distinct series.

In [10]:
fingerprint = daily.groupby("lgd_district_code").agg(
    met=("t2m_c", lambda s: hash(tuple(s.round(2).fillna(-1)))),
    sun=("ghi_kwh_m2_day", lambda s: hash(tuple(s.round(3).fillna(-1)))),
)
print("distinct MERRA-2 temperature series:", fingerprint["met"].nunique())
print("distinct CERES irradiance series:   ", fingerprint["sun"].nunique())
shared = fingerprint.merge(spine[["lgd_district_code", "district_name"]], on="lgd_district_code")
groups = shared.groupby("met")["district_name"].apply(sorted)
groups[groups.str.len() > 1].sort_values().reset_index(drop=True)

distinct MERRA-2 temperature series:

 58
distinct CERES irradiance series:    26


0                           [Agra, Firozabad]
1                        [Amethi, Pratapgarh]
2                 [Amroha, Moradabad, Rampur]
3                            [Ayodhya, Basti]
4                           [Baghpat, Meerut]
5            [Bahraich, Balrampur, Shrawasti]
6                       [Bara Banki, Lucknow]
7                       [Chandauli, Varanasi]
8                          [Etawah, Mainpuri]
9     [Gautam Buddha Nagar, Ghaziabad, Hapur]
10              [Gorakhpur, Sant Kabir Nagar]
11                      [Kanpur Nagar, Unnao]
12                   [Kushinagar, Mahrajganj]
13                    [Muzaffarnagar, Shamli]
Name: district_name, dtype: object

## 6. Does it vary enough between districts to matter? (a first look, not the EDA write-up)

Per district: mean annual GHI, mean annual rainfall and its coefficient of variation,
mean days per year with Tmax ≥ 40 °C. The variable-by-variable exploration (EDA step 4)
is Sage's; this only checks the data can support it.

In [11]:
d = daily.assign(year=daily["date"].dt.year)
annual = d.groupby(["lgd_district_code", "year"]).agg(
    ghi=("ghi_kwh_m2_day", "mean"),
    rain_mm=("precip_mm", "sum"),
    hot_days=("t2m_max_c", lambda s: (s >= 40).sum()),
)
per_district = annual.groupby("lgd_district_code").agg(
    ghi_kwh_m2_day=("ghi", "mean"),
    rain_mm_per_year=("rain_mm", "mean"),
    rain_cv=("rain_mm", lambda s: s.std() / s.mean()),
    hot_days_per_year=("hot_days", "mean"),
)
spread = per_district.describe().T[["min", "50%", "max"]]
spread["max/min"] = spread["max"] / spread["min"]
spread.round(2)

,min,50%,max,max/min
ghi_kwh_m2_day,4.66,4.83,5.16,1.11
rain_mm_per_year,647.17,889.05,1398.04,2.16
rain_cv,0.18,0.21,0.28,1.57
hot_days_per_year,37.52,65.68,75.52,2.01


## 7. Write outputs and read them back

In [12]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
pts_out = pts[
    [
        "lgd_district_code",
        "district_name",
        "wikidata_district",
        "hq_label",
        "point_kind",
        "lat",
        "lon",
        "hq_km_to_district_point",
    ]
]
pts_out.to_parquet(OUT_DIR / "hq_points.parquet", index=False)
daily.to_parquet(OUT_DIR / "power_daily.parquet", index=False)
check = duckdb.sql(f"""
    select (select count(*) from '{(OUT_DIR / "hq_points.parquet").as_posix()}') as n_points,
           (select count(distinct lgd_district_code)
              from '{(OUT_DIR / "power_daily.parquet").as_posix()}') as n_districts,
           (select count(*) from '{(OUT_DIR / "power_daily.parquet").as_posix()}') as n_rows
""").df()
check

,n_points,n_districts,n_rows
0,75,75,684825


In [13]:
assert check.loc[0, "n_points"] == 75 and check.loc[0, "n_districts"] == 75
assert check.loc[0, "n_rows"] == len(daily)

## 8. Findings

Filled in from the run on 2026-11-24 (numbers are in the outputs above).

1. **All 75 districts found in Wikidata by LGD code** (P12746), so no name matching. 73
   points are the HQ. Two fall back to the district item's own coordinate, flagged in
   `point_kind`: Siddharthnagar (HQ item Navgarh has no coordinates) and **Amethi**, whose
   HQ "Gauriganj" is linked to the wrong Wikidata item (a Gauriganj at 87.7° E, outside
   UP). Some items carry two coordinate statements; the median is used. → DQ-008.
2. **NASA POWER answered for all 75 points**, 2001–2025 daily, 6 parameters, with
   negligible missing values.
3. **Grid cells are coarse** (§5): the 75 points give only **58** distinct MERRA-2
   (temperature, rain) series and **26** distinct CERES irradiance series, so groups of
   neighbouring districts (e.g. Amroha–Moradabad–Rampur) have identical climate. Fine for
   flags and context, weak as a fine-grained ranker. → DQ-009.
4. **Variation** (§6), 2001–2025 means: irradiance 4.66–5.16 kWh/m²/day (max/min 1.11), so
   solar yield barely separates districts; rainfall 647–1,398 mm/yr (×2.2), its CV
   0.18–0.28, and days ≥ 40 °C 38–76 per year (×2.0) vary much more. That matches the design
   doc's plan to show climate as risk flags rather than a cost term.
5. **Licences**: Wikidata CC0; NASA POWER CC BY 4.0 (attribution in `docs/data-sources.md`).